# 🔥 Load Testing — dev-unonomercy.in
**Tool:** Locust (industry-standard load testing framework)

This notebook simulates realistic concurrent users hitting your dev server — including HTTP and WebSocket (Socket.IO) connections.

---

## Step 1 — Install Dependencies

In [ ]:
!pip install locust websocket-client python-socketio requests -q
print('✅ All dependencies installed')

## Step 2 — Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values
# ============================================================

TARGET_URL    = 'https://dev-unonomercy.in'   # Dev server
USERS         = 100    # Peak concurrent virtual users
SPAWN_RATE    = 10     # Users spawned per second (ramp-up speed)
RUN_TIME      = '3m'   # Total test duration (e.g. '2m', '5m', '10m')
SOCKET_ENABLED = True  # Set False if your site has no Socket.IO

# Routes to test (add your real routes here)
HTTP_ROUTES = [
    '/',
    '/login',
    '/dashboard',
    '/api/status',
    '/api/users',
]

print(f'✅ Config loaded')
print(f'   Target  : {TARGET_URL}')
print(f'   Users   : {USERS} peak, {SPAWN_RATE}/sec ramp')
print(f'   Duration: {RUN_TIME}')
print(f'   Socket  : {"Enabled" if SOCKET_ENABLED else "Disabled"}')

## Step 3 — Write the Locust Test File

In [ ]:
locustfile_content = '''
import random
import time
import json
import threading
from locust import HttpUser, task, between, events
import websocket

# ── Routes to load-test ─────────────────────────────────────
HTTP_ROUTES = ''' + str(HTTP_ROUTES) + '''
SOCKET_ENABLED = ''' + str(SOCKET_ENABLED) + '''

# ── HTTP User ───────────────────────────────────────────────
class WebsiteUser(HttpUser):
    """Simulates a real user browsing the site."""
    wait_time = between(1, 4)   # Think time between requests (seconds)

    def on_start(self):
        """Called once when a virtual user starts."""
        self.headers = {
            'Accept': 'text/html,application/xhtml+xml,application/json',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'User-Agent': 'Mozilla/5.0 (LoadTest-Bot)',
        }
        self.socket_thread = None
        if SOCKET_ENABLED:
            self._start_socket()

    # ── HTTP Tasks ──────────────────────────────────────────
    @task(5)
    def visit_random_page(self):
        route = random.choice(HTTP_ROUTES)
        with self.client.get(route, headers=self.headers, catch_response=True) as r:
            if r.status_code in [200, 201, 304]:
                r.success()
            elif r.status_code == 404:
                r.failure(f'404 Not Found: {route}')
            elif r.status_code >= 500:
                r.failure(f'Server error {r.status_code} on {route}')

    @task(3)
    def visit_homepage(self):
        with self.client.get('/', headers=self.headers, catch_response=True) as r:
            if r.elapsed.total_seconds() > 3:
                r.failure(f'Slow response: {r.elapsed.total_seconds():.2f}s')
            elif r.status_code == 200:
                r.success()

    @task(2)
    def api_ping(self):
        with self.client.get('/api/status', headers=self.headers, catch_response=True) as r:
            if r.status_code in [200, 404]:  # 404 ok if route doesn't exist
                r.success()

    # ── Socket.IO Simulation ────────────────────────────────
    def _start_socket(self):
        """Opens a WebSocket connection in a background thread."""
        base = self.host.replace('https://', 'wss://').replace('http://', 'ws://')
        ws_url = f"{base}/socket.io/?EIO=4&transport=websocket"

        def run():
            try:
                ws = websocket.create_connection(ws_url, timeout=5)
                ws.send('40')  # Socket.IO connect
                time.sleep(random.uniform(10, 30))  # Simulate session duration
                ws.send('41')  # Socket.IO disconnect
                ws.close()
            except Exception:
                pass  # Socket failures are non-blocking

        self.socket_thread = threading.Thread(target=run, daemon=True)
        self.socket_thread.start()

    def on_stop(self):
        pass

# ── Event Hooks for CSV Logging ─────────────────────────────
import csv
import os

csv_file = open('/content/load_test_results.csv', 'w', newline='')
csv_writer = csv.writer(csv_file)
csv_writer.writerow(['timestamp', 'route', 'method', 'status', 'response_time_ms', 'success'])

@events.request.add_listener
def on_request(request_type, name, response_time, response_length, response, exception, **kw):
    success = exception is None and (response is None or response.status_code < 400)
    csv_writer.writerow([
        time.strftime('%H:%M:%S'),
        name,
        request_type,
        response.status_code if response else 'ERR',
        round(response_time),
        success
    ])
    csv_file.flush()
'''

with open('/content/locustfile.py', 'w') as f:
    f.write(locustfile_content)

print('✅ locustfile.py written')

## Step 4 — Run the Load Test

> This runs headless (no browser UI). Results stream to console and save to CSV.

In [ ]:
import subprocess

cmd = [
    'locust',
    '-f', '/content/locustfile.py',
    '--headless',
    '--host', TARGET_URL,
    '-u', str(USERS),
    '-r', str(SPAWN_RATE),
    '-t', RUN_TIME,
    '--only-summary',
    '--csv', '/content/locust_report',
    '--html', '/content/locust_report.html',
    '--loglevel', 'WARNING',
]

print(f'🚀 Starting load test against {TARGET_URL}')
print(f'   {USERS} users | {SPAWN_RATE}/sec ramp | {RUN_TIME} duration')
print('   Streaming results...\n')

result = subprocess.run(cmd, capture_output=False, text=True)
print('\n✅ Test complete!')

## Step 5 — View Results Summary

In [ ]:
import pandas as pd
import os

# Read Locust's built-in CSV stats
stats_file = '/content/locust_report_stats.csv'
if os.path.exists(stats_file):
    df = pd.read_csv(stats_file)
    print('\n📊 LOAD TEST RESULTS SUMMARY')
    print('=' * 70)
    cols = ['Name', 'Request Count', 'Failure Count', 'Average Response Time',
            'Min Response Time', 'Max Response Time', 'Requests/s']
    available = [c for c in cols if c in df.columns]
    print(df[available].to_string(index=False))

    total = df[df['Name'] == 'Aggregated'] if 'Aggregated' in df['Name'].values else df.tail(1)
    print('\n📈 OVERALL')
    print(f'   Total Requests : {int(total["Request Count"].values[0]):,}')
    print(f'   Failures       : {int(total["Failure Count"].values[0]):,}')
    print(f'   Avg Response   : {float(total["Average Response Time"].values[0]):.0f} ms')
    print(f'   Requests/sec   : {float(total["Requests/s"].values[0]):.1f}')
    fail_rate = float(total['Failure Count'].values[0]) / max(float(total['Request Count'].values[0]), 1) * 100
    print(f'   Failure Rate   : {fail_rate:.1f}%')

    if fail_rate > 10:
        print('\n⚠️  HIGH failure rate — your server may be struggling under this load.')
    elif fail_rate > 2:
        print('\n⚠️  Some failures detected — check server logs.')
    else:
        print('\n✅ Low failure rate — server handling load well.')
else:
    print('Stats file not found — check if the test ran correctly.')

## Step 6 — Plot Response Time & Throughput

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

hist_file = '/content/locust_report_stats_history.csv'
if os.path.exists(hist_file):
    hist = pd.read_csv(hist_file)
    hist = hist[hist['Name'] == 'Aggregated'] if 'Aggregated' in hist['Name'].values else hist

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f'Load Test — {TARGET_URL}', fontsize=14, fontweight='bold')

    # Response time
    ax1 = axes[0, 0]
    if '50%' in hist.columns:
        ax1.plot(hist['50%'], label='p50 (median)', color='steelblue')
    if '95%' in hist.columns:
        ax1.plot(hist['95%'], label='p95', color='orange')
    if '99%' in hist.columns:
        ax1.plot(hist['99%'], label='p99', color='red', linestyle='--')
    ax1.set_title('Response Time (ms)')
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('ms')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Requests/sec
    ax2 = axes[0, 1]
    if 'Requests/s' in hist.columns:
        ax2.fill_between(range(len(hist)), hist['Requests/s'], alpha=0.4, color='green')
        ax2.plot(hist['Requests/s'], color='green')
    ax2.set_title('Throughput (requests/sec)')
    ax2.set_xlabel('Sample')
    ax2.set_ylabel('req/s')
    ax2.grid(True, alpha=0.3)

    # User count
    ax3 = axes[1, 0]
    if 'User count' in hist.columns:
        ax3.fill_between(range(len(hist)), hist['User count'], alpha=0.4, color='purple')
        ax3.plot(hist['User count'], color='purple')
    ax3.set_title('Concurrent Users')
    ax3.set_xlabel('Sample')
    ax3.set_ylabel('Users')
    ax3.grid(True, alpha=0.3)

    # Failures/sec
    ax4 = axes[1, 1]
    if 'Failures/s' in hist.columns:
        ax4.fill_between(range(len(hist)), hist['Failures/s'], alpha=0.4, color='red')
        ax4.plot(hist['Failures/s'], color='red')
    ax4.set_title('Failure Rate (failures/sec)')
    ax4.set_xlabel('Sample')
    ax4.set_ylabel('failures/s')
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/load_test_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Chart saved to /content/load_test_chart.png')
else:
    print('History file not found. Run Step 4 first.')

## Step 7 — Download Reports

In [ ]:
from google.colab import files
import os

files_to_download = [
    '/content/locust_report.html',      # Full HTML report
    '/content/locust_report_stats.csv', # Stats CSV
    '/content/load_test_results.csv',   # Per-request log
    '/content/load_test_chart.png',     # Charts
]

for f in files_to_download:
    if os.path.exists(f):
        files.download(f)
        print(f'📥 Downloaded: {f}')
    else:
        print(f'⚠️  Not found: {f}')

---
## 📝 How to Interpret Results

| Metric | Good | Warning | Critical |
|--------|------|---------|----------|
| Avg Response Time | < 500ms | 500ms–2s | > 2s |
| p95 Response Time | < 1s | 1s–5s | > 5s |
| Failure Rate | < 1% | 1%–5% | > 5% |
| Requests/sec | Stable | Dropping | Falling fast |

## ⚙️ Tuning
- Increase `USERS` gradually (50 → 100 → 200) to find the breaking point
- Add your real API routes in `HTTP_ROUTES`
- Set `SOCKET_ENABLED = False` if you don't use Socket.IO
- Change `RUN_TIME` to `'10m'` for a longer sustained test